In [2]:
from ultralytics import YOLO
import torch
import numpy as np
print(torch.backends.mps.is_available())  # Должно быть True

True


In [ ]:
model = YOLO("/Users/igorzolotyh/SafeVision/FaceSpeech/runs/train/exp8/weights/best.pt")

model.train(
    data="/Users/igorzolotyh/SafeVision/FaceSpeech/second_tongue_detection/data.yaml",
    epochs=20,
    imgsz=256,
    batch=32,
    device="mps",  # NPU MacBook
    project="runs/train",
    name="exp"
)

In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt

%matplotlib inline

model = YOLO("/Users/igorzolotyh/SafeVision/FaceSpeech/runs/train/exp8/weights/best.pt")

image_path = "/Users/igorzolotyh/SafeVision/FaceSpeech/images/crop.png"
results = model(image_path)

img = cv2.imread(image_path)
for r in results:
    boxes = r.boxes.xyxy.cpu().numpy()  # [x_min, y_min, x_max, y_max]
    for box in boxes:
        x_min, y_min, x_max, y_max = map(int, box[:4])
        #cv2.rectangle(img, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

        cv2.rectangle(img, (x_min, y_max-15), (x_max, y_max), (200, 200, 0), 2)

        

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.imshow(img_rgb)
plt.axis("off")
plt.show()

In [ ]:
model.export(format="tflite")  # creates 'yolo11n_float32.tflite'

tflite_model = YOLO("yolo11n_float32.tflite")

In [ ]:
from onnx_tf.backend import prepare
import onnx

onnx_model = onnx.load("/Users/igorzolotyh/SafeVision/FaceSpeech/runs/train/exp2/weights/best.onnx")
tf_rep = prepare(onnx_model)
tf_rep.export_graph("saved_model")


In [5]:
import tensorflow as tf
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Загружаем TFLite модель
interpreter = tf.lite.Interpreter(model_path="/Users/igorzolotyh/SafeVision/FaceSpeech/runs/train/exp2/weights/best_saved_model/best_float32.tflite")
interpreter.allocate_tensors()

# Получаем информацию о входах/выходах
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Загружаем и подготавливаем изображение
def preprocess_image(image_path, target_size=(256, 256)):
    img = Image.open(image_path)
    img = img.resize(target_size)  # Приводим изображение к нужному размеру
    img = np.array(img).astype(np.float32)
    img = np.expand_dims(img, axis=0)  # Добавляем размер батча
    return img

# Пример изображения
image_path = "/Users/igorzolotyh/SafeVision/FaceSpeech/images/4.jpeg"  # Заменить на путь к твоему изображению
input_image = preprocess_image(image_path)

# Получаем входной тензор
interpreter.set_tensor(input_details[0]['index'], input_image)

# Запускаем инференс
interpreter.invoke()

# Получаем результат (например, для YOLO это будут боксы и классы)
output_data = interpreter.get_tensor(output_details[0]['index'])

# Выводим результат
print("Model output:", output_data)

# Визуализируем результаты (например, для детекции объектов)
def visualize_detection(image_path, output_data):
    # Загружаем исходное изображение для отображения
    img = Image.open(image_path)
    img = np.array(img)

    # Предположим, что output_data это [x1, y1, x2, y2, confidence, class_id] для каждого объекта
    for detection in output_data:
        # Получаем координаты бокса
        ymin, xmin, ymax, xmax = detection[0:4]
        confidence = detection[4]
        class_id = int(detection[5])

        # Нарисуем прямоугольник на изображении
        plt.rectangle((xmin, ymin), (xmax, ymax), outline="red", width=2)
        plt.text(xmin, ymin, f"Class {class_id} Conf: {confidence:.2f}", color="white")

    # Отображаем изображение
    plt.imshow(img)
    plt.show()

# Визуализируем результат
visualize_detection(image_path, output_data)


/opt/anaconda3/envs/safevision/lib/python3.10/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


Model output: [[[   0.040747    0.016457    0.043063 ...      0.6875        0.75       0.875]
  [   0.049585    0.053347    0.049765 ...      0.9375      0.9375      0.8125]
  [   0.071775    0.064167    0.070124 ...        0.75       0.625       0.625]
  [   0.068883    0.075516    0.068428 ...         0.5         0.5        1.25]
  [ 3.3402e-35           0  3.9624e-37 ...           1           1     0.36389]]]


IndexError: index 5 is out of bounds for axis 0 with size 5

In [ ]:
import numpy as np
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt

# Параметры
MODEL_PATH = "/Users/igorzolotyh/SafeVision/FaceSpeech/runs/train/exp2/weights/best_saved_model/best_float32.tflite"
IMG_PATH = "/Users/igorzolotyh/SafeVision/FaceSpeech/images/istockphoto-111824137-612x612.jpg"
IMG_SIZE = 256
CONF_THRESHOLD = 0.01

# Загрузка модели
interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

# Получение деталей входа и выхода
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print("Input shape:", input_details[0]['shape'])
print("Output shape:", output_details[0]['shape'])

# Загрузка и предобработка изображения
img = cv2.imread(IMG_PATH)
img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
img_input = img_resized.astype(np.float32) / 255.0
img_input = np.expand_dims(img_input, axis=0)

# Установка входного тензора
interpreter.set_tensor(input_details[0]['index'], img_input)

# Выполнение инференса
interpreter.invoke()

# Получение выхода
output_data = interpreter.get_tensor(output_details[0]['index'])  # [5, 1344]
print("Output data shape:", output_data.shape)


Input shape: [  1 256 256   3]
Output shape: [   1    5 1344]
Output data shape: (1, 5, 1344)


In [1]:
import numpy as np
import tensorflow as tf
from PIL import Image

# Загрузка TFLite модели
interpreter = tf.lite.Interpreter(model_path="/Users/igorzolotyh/SafeVision/FaceSpeech/runs/train/exp5/weights/best_saved_model/best_float32.tflite")
interpreter.allocate_tensors()

# Получение входных и выходных тензоров
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()


/opt/anaconda3/envs/safevision/lib/python3.10/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [5]:
image_path = "/Users/igorzolotyh/SafeVision/FaceSpeech/images/2.jpeg"
image = Image.open(image_path).resize((640, 640))
input_data = np.array(image, dtype=np.float32) / 255.0 
input_data = np.expand_dims(input_data, axis=0)

# Установка входных данных
interpreter.set_tensor(input_details[0]["index"], input_data)

# 3. Выполнение инференса
interpreter.invoke()

# Получение выходных данных
output_data = interpreter.get_tensor(output_details[0]["index"])  # [1, 5, 1344]
output_data = output_data[0]  # Убираем batch dimension: [5, 1344]

# 4. Разбор выхода
# Предполагаем, что выход: [x_center, y_center, width, height, confidence]
boxes = output_data[:4, :].T  # [1344, 4] - координаты [x, y, w, h]
scores = output_data[4, :].T  # [1344] - уверенности

# Преобразование координат в [x_min, y_min, x_max, y_max] для NMS
x_center, y_center, w, h = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
x_min = x_center - w / 2
y_min = y_center - h / 2
x_max = x_center + w / 2
y_max = y_center + h / 2
boxes_xyxy = np.stack([x_min, y_min, x_max, y_max], axis=1)  # [1344, 4]

# 5. Применение NMS
selected_indices = tf.image.non_max_suppression(
    boxes_xyxy, scores, max_output_size=100, iou_threshold=0.2, score_threshold=0.2
)
selected_boxes = boxes_xyxy[selected_indices]
selected_scores = scores[selected_indices]

# 6. Вывод результатов
print("Найдено объектов:", len(selected_boxes))
for i, (box, score) in enumerate(zip(selected_boxes, selected_scores)):
    print(f"Объект {i}: box={box}, confidence={score:.4f}")

Найдено объектов: 0


In [ ]:
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# Загружаем и масштабируем оригинальную картинку для визуализации
orig_image = cv2.imread(image_path)
orig_image = cv2.resize(orig_image, (640, 640))

# Рисуем рамки
for box in selected_boxes:
    x_min, y_min, x_max, y_max = box * 640  # Преобразуем нормализованные координаты
    x_min, y_min, x_max, y_max = map(int, [x_min, y_min, x_max, y_max])
    cv2.rectangle(orig_image, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

# Переводим BGR → RGB для отображения в matplotlib
img_rgb = cv2.cvtColor(orig_image, cv2.COLOR_BGR2RGB)

# Отображаем
plt.figure(figsize=(6, 6))
plt.imshow(img_rgb)
plt.title("TFLite Detection Result")
plt.axis("off")
plt.show()


In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

def detect_tongue_tip(image, tongue_box):
    x_min, y_min, x_max, y_max = map(int, tongue_box)
    
    roi = image[y_min:y_max, x_min:x_max]
    if roi.size == 0:  
        return None
    
    roi_hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    
    search_height = int(0.2 * (y_max - y_min))
    search_y_min = y_max - y_min - search_height
    search_roi_hsv = roi_hsv[search_y_min:, :]
    
    lower_tongue = np.array([0, 50, 100])
    upper_tongue = np.array([30, 255, 255])
    mask = cv2.inRange(search_roi_hsv, lower_tongue, upper_tongue)
    
    y_coords, x_coords = np.where(mask == 255)
    if len(x_coords) > 0:
        max_y_idx = np.argmax(y_coords)
        tip_local = (x_coords[max_y_idx], y_coords[max_y_idx])
        tip = (tip_local[0] + x_min, tip_local[1] + search_y_min + y_min)
    else:
        tip = ((x_min + x_max) // 2, y_max)
    
    return tip, mask

model = YOLO("/Users/igorzolotyh/SafeVision/FaceSpeech/runs/train/exp8/weights/best.pt")

image_path = "/Users/igorzolotyh/SafeVision/FaceSpeech/images/5.jpeg"
results = model(image_path)

img = cv2.imread(image_path)
if img is None:
    print("Ошибка: не удалось загрузить изображение")
    exit()

for r in results:
    boxes = r.boxes.xyxy.cpu().numpy()  # [x_min, y_min, x_max, y_max]
    for box in boxes:
        x_min, y_min, x_max, y_max = map(int, box[:4])
        cv2.rectangle(img, (x_min, y_max-15), (x_max, y_max), (200, 200, 0), 2)
        
        # Кончик
        tip, mask = detect_tongue_tip(img, (x_min, y_min, x_max, y_max))
        if tip is not None:
            cv2.circle(img, (int(tip[0]), int(tip[1])), 5, (0, 255, 0), -1)

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.imshow(img_rgb)
plt.axis("off")
plt.show()

In [ ]:
plt.imshow(mask, cmap='gray')